# Lab 6: A/B Testing and Confidence Intervals

In [ ]:
# Run this cell to set up the notebook, but please don't change it.

import numpy as np
from datascience import *

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## 1. A/B Testing

A/B testing is a form of hypothesis testing that allows you to make comparisons between two distributions.

You'll almost never be explicitly asked to perform an A/B test. Make sure you can identify situations where the test is appropriate and know how to correctly implement each step.  

**Question 1.1.** The following statements are the unordered steps of an A/B hypothesis test:

1. Find the value of the observed test statistic

2. Choose a test statistic (typically the difference in means between two categories)

3. Shuffle the labels of the original sample, find your simulated test statistic, and repeat many times

4. Calculate the p-value based off your observed and simulated test statistics

5. Define a null and alternative model

6. Use the p-value and p-value cutoff to draw a conclusion about the null hypothesis

Make an array called `ab_test_order` that contains the correct order of an A/B test, where the first item of the array is the first step of an A/B test and the last item of the array is the last step of an A/B test

In [ ]:
ab_test_order = ...

**Question 1.2.** If the null hypothesis of an A/B test is correct, should the order of labels affect the differences in means between each group? Why do we shuffle labels in an A/B test? 

*Write your answer here, replacing this text.*

## 2. Murder Rates and the Death Penalty

Punishment for crime has many [philosophical justifications](https://plato.stanford.edu/entries/legal-punishment/).  An important one is that fear of punishment may *deter* people from committing crimes.

In the United States, some jurisdictions execute people who are convicted of particularly serious crimes, such as murder.  This punishment is called the *death penalty* or *capital punishment*.  The death penalty is controversial, and deterrence has been one focal point of the debate.  There are other reasons to support or oppose the death penalty, but in this project we'll focus on deterrence.

The key question about deterrence is:

> Through our exploration, does instituting a death penalty for murder actually reduce the number of murders?

You might have a strong intuition in one direction, but the evidence turns out to be surprisingly complex.  Different sides have variously argued that the death penalty has no deterrent effect and that each execution prevents 8 murders, all using statistical arguments!  We'll try to come to our own conclusion.

The main data source for this lab comes from a paper by three researchers, Dezhbakhsh, Rubin, and Shepherd.  The dataset contains rates of various violent crimes for every year 1960-2003 (44 years) in every US state.  The researchers compiled the data from the FBI's Uniform Crime Reports.

Since crimes are committed by people, not states, we need to account for the number of people in each state when we're looking at state-level data.  Murder rates are calculated as follows:

$$\text{murder rate for state X in year Y} = \frac{\text{number of murders in state X in year Y}}{\text{population in state X in year Y}}*100000$$

(Murder is rare, so we multiply by 100,000 just to avoid dealing with tiny numbers.)

### A Natural Experiment

In order to attempt to investigate the causal relationship between the death penalty and murder rates, we're going to take advantage of a *natural experiment*.  A natural experiment happens when something other than experimental design applies a treatment to one group and not to another (control) group, and we have some hope that the treatment and control groups don't have any other systematic differences.

Our natural experiment is this: in 1972, a Supreme Court decision called *Furman v. Georgia* banned the death penalty throughout the US.  Suddenly, many states went from having the death penalty to not having the death penalty.

As a first step, let's see how murder rates changed before and after the court decision.  We'll define the test as follows:

> **Population:** All the states that had the death penalty before the 1972 abolition.  (There is no control group for the states that already lacked the death penalty in 1972, so we must omit them.)  This includes all US states **except** Alaska, Hawaii, Maine, Michigan, Wisconsin, and Minnesota.

> **Treatment group:** The states in that population, in 1973 (the year after 1972).

> **Control group:** The states in that population, in 1971 (the year before 1972).

> **Null hypothesis:** Murder rates in 1971 and 1973 come from the same distribution.

> **Alternative hypothesis:** Murder rates were higher in 1973 than they were in 1971.

Our alternative hypothesis is related to our suspicion that murder rates increase when the death penalty is eliminated.  

Run the cell below to import the table `change_in_murder_rates` that contains each state's population, murder rate, and whether or not that state had the death penalty for both 1971 and 1973. 

In [ ]:
change_in_murder_rates = Table.read_table('/home/jovyan/shared/Data/change_in_murder_rates.csv')
change_in_murder_rates

Run the cell below to view the distribution of murder rates during the pre-ban and post-ban time periods.

In [ ]:
change_in_murder_rates.hist('Murder Rate', group = 'Death Penalty')

**Question 2.1.** Create a table `rate_means` that contains the average murder rates for the states that had the death penalty and the states that didn't have the death penalty. It should have two columns: one indicating if the penalty was in place, and one that contains the average murder rate for each group.

In [ ]:
rate_means = ...
rate_means

**Question 2.2.** We want to figure out if there is a difference between the distribution of murder rates in 1971 and 1973. Specifically, we want to test if murder rates were higher in 1973 than they were in 1971. 

What should the test statistic be? How does it help us differentiate whether the data supports the null and alternative? 

*Write your answer here, replacing this text.*

**Question 2.3.** Set `observed_difference` to the observed test statistic using the `rate_means` table 

In [ ]:
observed_difference = ...
observed_difference

**Question 2.4.** Given a table like `change_in_murder_rates`, a value column `label`, and a group column `group_label`, write a function that calculates the appropriate test statistic.

In [ ]:
def find_test_stat(table, labels_col, values_col):
    ...

find_test_stat(change_in_murder_rates, "Death Penalty", "Murder Rate")

When we run a simulation for A/B testing, we resample by shuffling the labels of the original sample. If the null hypothesis is true and the murder rate distributions are the same, we expect that the difference in mean murder rates will be not change when "Death Penalty" labels are changed.

**Question 2.5:** Write a function `simulate_and_test_statistic` to compute one trial of our A/B test. Your function should run a simulation and return a test statistic.

In [ ]:
def simulate_and_test_statistic(table, labels_col, values_col):
    ...

simulate_and_test_statistic(change_in_murder_rates, "Death Penalty", "Murder Rate")

**Question 2.6.** Simulate 5000 trials of our A/B test and store the test statistics in an array called `differences`

In [ ]:
# This cell might take a couple seconds to run
differences = make_array()

...
                                                 
differences

Run the cell below to view a histogram of your simulated test statistics plotted with your observed test statistic

In [ ]:
Table().with_column('Difference Between Group Means', differences).hist()
plots.plot([observed_difference, observed_difference], [0, 0.5], color='red', lw=2);

**Question 2.7.** Find the p-value for your test and assign it to `empirical_P`

In [ ]:
empirical_P = ...
empirical_P

**Question 2.8.** Using a 5% P-value cutoff, draw a conclusion about the null and alternative hypotheses. Describe your findings using simple, non-technical language. What does your analysis tell you about murder rates after the death penalty was suspended? What can you claim about causation from your statistical analysis?

*Write your answer here, replacing this text.*

## 3. Election Polls


East Carolina University [conducted a poll](https://surveyresearch-ecu.reportablenews.com/pr/nc-senate-budd-maintains-lead-over-beasley-as-election-day-nears) from November 1st through 3rd in 2022 of likely voters in the U.S. Senate election in North Carolina. The data for the poll were drawn from a cellphone sample of SMS-to-web provided by Consensus Strategies, an Interactive Voice Response (IVR) system of landlines only, and an online panel provided by Bovitz. To ensure the sample was representative of the midterm electorate, the data were weighted by gender, age, education, race, region, and election modeling based on previous statewide elections in North Carolina.

The poll showed Republican Ted Budd leading Democrat Cheri Beasley, 52% to 46%, among likely voters, with just 1% of the poll respondents reporting they are undecided (and the remaining 1% expressing support for some other candidate).

|Responses  | Percentage|
|:------------:|:------------:|
|Ted Budd | 52% |
|Cheri Beasley | 46% |
|Undecided | 1% |
|Some other candidate | 1% |

We will attempt to estimate the corresponding *parameters*, or the percentage of the votes that each response will receive from the entire population (the entire population is all NC voters). We will use confidence intervals to compute a range of values that reflects the uncertainty of our estimates. Now that the election has occurred, we could compare this with the actual result.

The table `poll2022` contains the results of the survey. The percentages above were rounded to the nearest percent.

In [ ]:
# Just run this cell
poll2022 = Table.read_table('/home/jovyan/shared/Data/poll2022.csv')
poll2022.group('Response').sort('count', True)

**Question 3.1.** Complete the function `one_resampled_percentage` below. It should return Cheri Beasley's **percentage** of votes after simulating one bootstrap sample of `tbl`.

**Note:** `tbl` will always be in the same format as `poll2022`.

In [ ]:
def one_resampled_percentage(tbl):
    ...

one_resampled_percentage(poll2022)

**Question 3.2.** Complete the `percentages_in_resamples` function such that it returns an array of 2500 bootstrapped estimates of the percentage of voters who will vote for Cheri Beasley. You should use the `one_resampled_percentage` function you wrote above.

In [ ]:
def percentages_in_resamples():
    percentage_Beasley = make_array()
    ...

In the following cell, we run the function you just defined, `percentages_in_resamples`, and create a histogram of the calculated statistic for the 2,500 bootstrap estimates of the percentage of voters who voted for Cheri Beasley. Based on what the original election poll percentages were, does the graph seem reasonable? 

In [ ]:
resampled_percentages = percentages_in_resamples()
Table().with_column('Estimated Percentage', resampled_percentages).hist("Estimated Percentage")

**Question 3.3.** Using the array `resampled_percentages`, find the values at the two edges of the middle 95% of the bootstrapped percentage estimates. (Compute the lower and upper ends of the interval, named `Beasley_lower_bound` and `Beasley_upper_bound`, respectively.)

In [ ]:
Beasley_lower_bound = ...
Beasley_upper_bound = ...
print("Bootstrapped 95% confidence interval for the percentage of Cheri Beasley voters in the population: [{:f}, {:f}]".format(Beasley_lower_bound, Beasley_upper_bound))

## 4. Interpreting Confidence Intervals


Dr. McLean computed the following **90%** confidence interval for the percentage of Cheri Beasley's voters: 

$$[43.9, 48.6]$$

**Question 4.1.** Can we say there is a 90% probability that the interval [43.9, 48.6] contains the true percentage of the population that votes for Cheri Beasley for US Senate in North Carolina (based on the sample from this poll, supposing that we do not know the election results)? Answer "yes" or "no" and explain your reasoning. 

*Note:* ambiguous answers using language like "sometimes" or "maybe" will not receive credit.

*Write your answer here, replacing this text.*

**Question 4.2.** Dr. McLean also created 75%, 82%, and 99% confidence intervals from the same sample, but he forgot to label which confidence interval represented which percentages! Match each confidence level (75%, 82%, 99%) with its corresponding interval in the cell below (e.g. __ % CI: [42.7, 50.0]$\rightarrow$ replace the blank with one of the three confidence levels). **Then**, explain your thought process and how you came up with your answers.

The intervals are below:

* [42.7, 50.0]
* [44.6, 48.0]
* [44.3, 48.2]

<!--
BEGIN QUESTION
name: q2_2
manual: true
-->
<!-- EXPORT TO PDF -->

*Write your answer here, replacing this text.*

**Question 4.3.** Suppose that we collect 600 new samples (each one a uniform random sample of likely voters in North Carolina of the same size as the original poll) from the population and created an 80% confidence interval from each one. Roughly how many of those 600 intervals do you expect will actually contain the true percentage of the population?

Assign your answer to `true_percentage_intervals`.

In [ ]:
true_percentage_intervals = ...
true_percentage_intervals